In [1]:
# ── Setup Session 2 : recréation de l'environnement depuis Session 1 ─────
import os
os.environ["JAVA_HOME"] = "/opt/homebrew/opt/openjdk@17"
os.environ["PATH"] = "/opt/homebrew/opt/openjdk@17/bin:" + os.environ.get("PATH", "")

import csv, io, time
from pathlib import Path
from pyspark.sql import SparkSession
from pyspark import StorageLevel

DATA_DIR       = Path("data")
VELIB_CSV      = DATA_DIR / "velib" / "raw" / "historique_stations.csv"
VELIB_PARQ_DIR = DATA_DIR / "velib" / "parquet"
STATIONS_CSV   = DATA_DIR / "velib" / "stations_info.csv"
METEO_CSV      = DATA_DIR / "meteo" / "paris_montsouris_horaire.csv"
OUTPUT_DIR     = DATA_DIR / "output"
APP_NAME       = "ClimaCity-Paris"
MASTER         = "local[*]"
SHUFFLE_PARTS  = 8

spark = (
    SparkSession.builder
    .appName(APP_NAME)
    .master(MASTER)
    .config("spark.sql.shuffle.partitions", SHUFFLE_PARTS)
    .config("spark.executor.memory", "16g")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
sc = spark.sparkContext
sc.setLogLevel("WARN")

COLONNES = [
    "horodatage", "capacite", "velos_disponibles",
    "bornettes_libres", "nom_station", "coordonnees", "is_installed"
]

def parse_ligne(line: str):
    try:
        champs = next(csv.reader(io.StringIO(line)))
        if len(champs) != 7:
            return None
        return {
            "horodatage"       : champs[0],
            "capacite"         : int(champs[1]),
            "velos_disponibles": int(champs[2]),
            "bornettes_libres" : int(champs[3]),
            "nom_station"      : champs[4],
            "coordonnees"      : champs[5],
            "is_installed"     : champs[6] == "True",
        }
    except Exception:
        return None

def ajouter_taux(record):
    record = dict(record)
    record["taux_occupation"] = record["velos_disponibles"] / record["capacite"]
    return record

raw_rdd   = sc.textFile(str(VELIB_CSV))
clean_rdd = raw_rdd.map(parse_ligne).filter(lambda r: r is not None)
step1     = clean_rdd.filter(lambda r: r["capacite"] > 0)
step2     = step1.map(ajouter_taux)
step3     = step2.filter(lambda r: r["taux_occupation"] < 0.10)

print(f"Spark {spark.version} prêt")
print(f"step3 : {step2.count():,} lignes")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/23 12:15:42 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 4.0.4 prêt
step3 : 5,267,323 lignes


---
# PARTIE 2 -- L'API DataFrame

## 2.1 Du RDD au DataFrame

L'API RDD est puissante mais verbeuse et peu optimisee : Spark ne sait pas ce que
contiennent les elements (ils sont des `object` Python opaques). L'API **DataFrame**
introduit un schema explicite, ce qui permet a l'optimiseur Catalyst de generer
un plan d'execution bien plus efficace.

### Creation depuis un RDD


In [2]:
from pyspark.sql import Row
from pyspark.sql.types import (
    StructType, StructField,
    IntegerType, StringType, DoubleType, BooleanType
)

# Schema adapté à nos colonnes réelles
schema_velib = StructType([
    StructField("horodatage",        StringType(),  nullable=True),
    StructField("capacite",          IntegerType(), nullable=True),
    StructField("velos_disponibles", IntegerType(), nullable=True),
    StructField("bornettes_libres",  IntegerType(), nullable=True),
    StructField("nom_station",       StringType(),  nullable=True),
    StructField("coordonnees",       StringType(),  nullable=True),
    StructField("is_installed",      BooleanType(), nullable=True),
    StructField("taux_occupation",   DoubleType(),  nullable=True),
])

# Convertir le RDD de dicts en RDD de Row, puis en DataFrame
row_rdd = step2.map(lambda d: Row(**d))
df_from_rdd = spark.createDataFrame(row_rdd, schema=schema_velib)

df_from_rdd.printSchema()
df_from_rdd.show(5, truncate=False)
print(f"Lignes : {df_from_rdd.count():,}  --  Partitions : {df_from_rdd.rdd.getNumPartitions()}")

root
 |-- horodatage: string (nullable = true)
 |-- capacite: integer (nullable = true)
 |-- velos_disponibles: integer (nullable = true)
 |-- bornettes_libres: integer (nullable = true)
 |-- nom_station: string (nullable = true)
 |-- coordonnees: string (nullable = true)
 |-- is_installed: boolean (nullable = true)
 |-- taux_occupation: double (nullable = true)

+-----------------+--------+-----------------+----------------+-----------------------------------+----------------+------------+-------------------+
|horodatage       |capacite|velos_disponibles|bornettes_libres|nom_station                        |coordonnees     |is_installed|taux_occupation    |
+-----------------+--------+-----------------+----------------+-----------------------------------+----------------+------------+-------------------+
|2020-11-26T12:59Z|35      |4                |5               |Benjamin Godard - Victor Hugo      |48.86598,2.27572|true        |0.11428571428571428|
|2020-11-26T12:59Z|55      |23    

---
## 2.2 Lecture directe depuis les fichiers Parquet

En pratique, on ne passe presque jamais par les RDD pour creer un DataFrame.
Spark peut lire directement de nombreux formats (CSV, JSON, Parquet, Delta, ORC...).

Le format **Parquet** est le format de facto pour les donnees analytiques :
- Stockage colonnaire (lecture partielle possible)
- Compression integree (Snappy, Zstandard...)
- Schema embarque dans les fichiers
- Partitionnement natif (Hive-style)


In [3]:
from pyspark.sql.functions import col, to_timestamp, year, month

  # 1. Parser le timestamp
df_avec_ts = df_from_rdd.withColumn(
      "timestamp",
      to_timestamp(col("horodatage"), "yyyy-MM-dd'T'HH:mmX")
  )
  
  # 2. Ajouter annee et mois
df_partitionne = df_avec_ts \
      .withColumn("annee", year(col("timestamp"))) \
      .withColumn("mois",  month(col("timestamp")))
  
  # 3. Écrire en Parquet partitionné
df_partitionne.write \
      .mode("overwrite") \
      .partitionBy("annee", "mois") \
      .parquet(str(VELIB_PARQ_DIR))

print(f"Parquet écrit dans {VELIB_PARQ_DIR}")

26/07/23 12:16:13 WARN MemoryManager: Total allocation exceeds 95,00% (1 020 054 720 bytes) of heap memory
Scaling row group sizes to 95,00% for 8 writers
26/07/23 12:16:13 WARN MemoryManager: Total allocation exceeds 95,00% (1 020 054 720 bytes) of heap memory
Scaling row group sizes to 84,44% for 9 writers
26/07/23 12:16:13 WARN MemoryManager: Total allocation exceeds 95,00% (1 020 054 720 bytes) of heap memory
Scaling row group sizes to 76,00% for 10 writers
26/07/23 12:16:14 WARN MemoryManager: Total allocation exceeds 95,00% (1 020 054 720 bytes) of heap memory
Scaling row group sizes to 84,44% for 9 writers
26/07/23 12:16:14 WARN MemoryManager: Total allocation exceeds 95,00% (1 020 054 720 bytes) of heap memory
Scaling row group sizes to 95,00% for 8 writers
26/07/23 12:16:14 WARN MemoryManager: Total allocation exceeds 95,00% (1 020 054 720 bytes) of heap memory
Scaling row group sizes to 84,44% for 9 writers
26/07/23 12:16:14 WARN MemoryManager: Total allocation exceeds 95,00%

Parquet écrit dans data/velib/parquet


In [4]:
# Lecture des fichiers Parquet pre-prepares (partitionnes par annee et mois)
# Spark detecte automatiquement le schema et les colonnes de partition
df_velib = spark.read.parquet(str(VELIB_PARQ_DIR))

df_velib.printSchema()
print(f"\nDimensions : {df_velib.count():,} lignes x {len(df_velib.columns)} colonnes")
print(f"Partitions : {df_velib.rdd.getNumPartitions()}")

root
 |-- horodatage: string (nullable = true)
 |-- capacite: integer (nullable = true)
 |-- velos_disponibles: integer (nullable = true)
 |-- bornettes_libres: integer (nullable = true)
 |-- nom_station: string (nullable = true)
 |-- coordonnees: string (nullable = true)
 |-- is_installed: boolean (nullable = true)
 |-- taux_occupation: double (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- annee: integer (nullable = true)
 |-- mois: integer (nullable = true)


Dimensions : 5,267,323 lignes x 11 colonnes
Partitions : 10


In [5]:
# Lecture selective d'une partition (predicate pushdown)
# Spark ne lira que les fichiers du dossier annee=2023/mois=06
df_juin_2021 = df_velib.filter("annee = 2021 AND mois = 01")
print(f"Juin 2021 : {df_juin_2021.count():,} lignes")

# Comparer avec une lecture complete puis filtre -- le plan d'execution est identique
# grace au predicate pushdown, mais on le verifie avec explain()
print("\n--- Plan d'execution (filtre apres lecture) ---")
df_velib.filter("annee = 2021 AND mois = 01").explain(mode="formatted")

Juin 2021 : 1,505,676 lignes

--- Plan d'execution (filtre apres lecture) ---
== Physical Plan ==
* ColumnarToRow (2)
+- Scan parquet  (1)


(1) Scan parquet 
Output [11]: [horodatage#48, capacite#49, velos_disponibles#50, bornettes_libres#51, nom_station#52, coordonnees#53, is_installed#54, taux_occupation#55, timestamp#56, annee#57, mois#58]
Batched: true
Location: InMemoryFileIndex [file:/Users/ms/Library/CloudStorage/SynologyDrive-cefedemaura/cours/hetic/spark_project/data/velib/parquet]
PartitionFilters: [isnotnull(annee#57), isnotnull(mois#58), (annee#57 = 2021), (mois#58 = 1)]
ReadSchema: struct<horodatage:string,capacite:int,velos_disponibles:int,bornettes_libres:int,nom_station:string,coordonnees:string,is_installed:boolean,taux_occupation:double,timestamp:timestamp>

(2) ColumnarToRow [codegen id : 1]
Input [11]: [horodatage#48, capacite#49, velos_disponibles#50, bornettes_libres#51, nom_station#52, coordonnees#53, is_installed#54, taux_occupation#55, timestamp#56, annee#57, 

---
## 2.3 Exploration et diagnostic des donnees

Avant tout traitement, il faut comprendre la structure et la qualite des donnees.
Spark propose des fonctions d'agregation statistiques integrees.


In [6]:
from pyspark.sql import functions as F

print("=== [2.3 - Etape 1/4] Statistiques descriptives (colonnes numeriques) ===")
df_velib.describe("capacite", "velos_disponibles", "bornettes_libres", "taux_occupation").show()
print("Etape 1/4 terminee.")

=== [2.3 - Etape 1/4] Statistiques descriptives (colonnes numeriques) ===


26/07/23 12:16:18 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------+------------------+-----------------+------------------+-------------------+
|summary|          capacite|velos_disponibles|  bornettes_libres|    taux_occupation|
+-------+------------------+-----------------+------------------+-------------------+
|  count|           5267323|          5267323|           5267323|            5267323|
|   mean| 31.67053283043398| 8.11338947696961|3.6443618513616878|0.24831664045539914|
| stddev|11.827212149609423|9.409048149962386| 3.232806624772183|0.23900153780243494|
|    min|                11|                0|                 0|                0.0|
|    max|                74|               73|                41| 1.0833333333333333|
+-------+------------------+-----------------+------------------+-------------------+

Etape 1/4 terminee.


In [7]:
print("=== [2.3 - Etape 2/4] Valeurs nulles par colonne ===")
df_velib.select([
    F.count(F.when(F.col(col_name).isNull(), col_name)).alias(col_name)
    for col_name in df_velib.columns
]).show()
print("Etape 2/4 terminee.")

=== [2.3 - Etape 2/4] Valeurs nulles par colonne ===
+----------+--------+-----------------+----------------+-----------+-----------+------------+---------------+---------+-----+----+
|horodatage|capacite|velos_disponibles|bornettes_libres|nom_station|coordonnees|is_installed|taux_occupation|timestamp|annee|mois|
+----------+--------+-----------------+----------------+-----------+-----------+------------+---------------+---------+-----+----+
|         0|       0|                0|               0|          0|          0|           0|              0|        0|    0|   0|
+----------+--------+-----------------+----------------+-----------+-----------+------------+---------------+---------+-----+----+

Etape 2/4 terminee.


In [8]:
print("=== [2.3 - Etape 3/4] Detection des anomalies ===")

# Stations avec capacite <= 0
print("  Calcul : capacite <= 0 ...")
n_cap_zero = df_velib.filter(F.col("capacite") <= 0).count()
print(f"  Snapshots avec capacite <= 0          : {n_cap_zero:,}")

# Snapshots avec plus de velos que la capacite (+2 de tolerance)
print("  Calcul : velos_disponibles > capacite + 2 ...")
n_debord = df_velib.filter(F.col("velos_disponibles") > F.col("capacite") + 2).count()
print(f"  Snapshots avec total > capacite + 2   : {n_debord:,}")

# Valeurs negatives
for col_name in ["velos_disponibles", "bornettes_libres"]:
    print(f"  Calcul : {col_name} < 0 ...")
    n_neg = df_velib.filter(F.col(col_name) < 0).count()
    print(f"  {col_name:<30} < 0 : {n_neg:,}")

print("Etape 3/4 terminee.")

=== [2.3 - Etape 3/4] Detection des anomalies ===
  Calcul : capacite <= 0 ...
  Snapshots avec capacite <= 0          : 0
  Calcul : velos_disponibles > capacite + 2 ...
  Snapshots avec total > capacite + 2   : 0
  Calcul : velos_disponibles < 0 ...
  velos_disponibles              < 0 : 0
  Calcul : bornettes_libres < 0 ...
  bornettes_libres               < 0 : 0
Etape 3/4 terminee.


In [10]:
print("=== [2.3 - Etape 4/4] Couverture temporelle (snapshots par mois) ===")
(
    df_velib
    .groupBy("annee", "mois")
    .count()
    .orderBy("annee", "mois")
    .show(50)
)
print("Etape 4/4 terminee.")

=== [2.3 - Etape 4/4] Couverture temporelle (snapshots par mois) ===
+-----+----+-------+
|annee|mois|  count|
+-----+----+-------+
| 2020|  11| 539932|
| 2020|  12|2795292|
| 2021|   1|1505676|
| 2021|   2| 426423|
+-----+----+-------+

Etape 4/4 terminee.


---
## 2.4 Nettoyage et construction des features

Nous allons maintenant construire la table `disponibilite` propre, telle que
definie dans le schema cible du projet.


In [11]:
from pyspark.sql.functions import (
    to_timestamp, col, when, lit,
    year, month, dayofweek, hour, greatest
)

# ── 1. Parsing du timestamp ───────────────────────────────────────────────────
df_clean = df_velib.withColumn(
    "timestamp_parsed",
    to_timestamp(col("horodatage"), "yyyy-MM-dd'T'HH:mmX")
)

# ── 2. Suppression des lignes avec timestamp invalide ───────────────────────
df_clean = df_clean.dropna(subset=["timestamp_parsed"])

# ── 3. Suppression des stations avec capacite invalide ──────────────────────
df_clean = df_clean.filter(col("capacite") > 0)

# ── 4. Correction des valeurs negatives (bornage a 0) ───────────────────────
df_clean = df_clean \
    .withColumn("velos_disponibles", greatest(col("velos_disponibles"), lit(0))) \
    .withColumn("bornettes_libres",  greatest(col("bornettes_libres"),  lit(0)))

# ── 5. Calcul du taux d'occupation ──────────────────────────────────────────
df_clean = df_clean.withColumn(
    "taux_occupation",
    col("velos_disponibles") / col("capacite")
)

# ── 6. Bornage du taux entre 0 et 1 ──────────────────────────────────────────
df_clean = df_clean.withColumn(
    "taux_occupation",
    when(col("taux_occupation") < 0, lit(0.0))
    .when(col("taux_occupation") > 1, lit(1.0))
    .otherwise(col("taux_occupation"))
)

# ── 7. Extraction de features temporelles ─────────────────────────────────────
df_clean = df_clean \
    .withColumn("annee",      year(col("timestamp_parsed"))) \
    .withColumn("mois",       month(col("timestamp_parsed"))) \
    .withColumn("jour_sem",   dayofweek(col("timestamp_parsed"))) \
    .withColumn("heure",      hour(col("timestamp_parsed"))) \
    .withColumn("is_weekend", col("jour_sem").isin([1, 7]))

print(f"Lignes apres nettoyage : {df_clean.count():,}")
df_clean.printSchema()
df_clean.show(3, truncate=True)

Lignes apres nettoyage : 5,267,323
root
 |-- horodatage: string (nullable = true)
 |-- capacite: integer (nullable = true)
 |-- velos_disponibles: integer (nullable = false)
 |-- bornettes_libres: integer (nullable = false)
 |-- nom_station: string (nullable = true)
 |-- coordonnees: string (nullable = true)
 |-- is_installed: boolean (nullable = true)
 |-- taux_occupation: double (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- annee: integer (nullable = true)
 |-- mois: integer (nullable = true)
 |-- timestamp_parsed: timestamp (nullable = true)
 |-- jour_sem: integer (nullable = true)
 |-- heure: integer (nullable = true)
 |-- is_weekend: boolean (nullable = true)

+-----------------+--------+-----------------+----------------+--------------------+----------------+------------+--------------------+-------------------+-----+----+-------------------+--------+-----+----------+
|       horodatage|capacite|velos_disponibles|bornettes_libres|         nom_station|     coo

In [12]:
# [EXERCICE]
# Ajoutez une colonne "statut" de type StringType avec les valeurs :
#   "vide"   si taux_occupation < 0.1
#   "plein"  si taux_occupation > 0.9
#   "normal" sinon
#
# Indice : utilisez F.when(...).when(...).otherwise(...)
# ──────────────────────────────────────────────────────────────────────────

df_clean = df_clean.withColumn(
    "statut",
    F.when(F.col("taux_occupation") < 0.1, "vide")
     .when(F.col("taux_occupation") > 0.9, "plein")
     .otherwise("normal")
)

df_clean.groupBy("statut").count().orderBy("statut").show()

+------+-------+
|statut|  count|
+------+-------+
|normal|3291063|
| plein|  23768|
|  vide|1952492|
+------+-------+



---
## 2.5 Chargement des donnees meteorologiques

La source meteorologique est le fichier SYNOP de Meteo-France pour la station
Paris-Montsouris, telechargeable librement sur data.gouv.fr.
Le format SYNOP est semi-structure et necessite quelques transformations.

Voici le format des colonnes pertinentes :

| Colonne SYNOP | Signification | Unite |
|---------------|---------------|-------|
| `AAAAMMJJHH`  | Horodatage    | UTC   |
| `T`           | Temperature   | K (Kelvin) |
| `U`           | Humidite relative | % |
| `FF`          | Vitesse du vent | m/s |
| `RR1`         | Precipitation | mm/h |
| `N`           | Nebulosity (couverture nuageuse) | octals (0-8) |


In [13]:
# Lecture du CSV meteo (format Open-Meteo, separateur point-virgule)
# Colonnes : horodatage;temperature;precipitations;vent_ms;humidite;pression
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

schema_meteo = StructType([
    StructField("horodatage",     StringType(), True),
    StructField("temperature",    DoubleType(), True),
    StructField("precipitations", DoubleType(), True),
    StructField("vent_ms",        DoubleType(), True),
    StructField("humidite",       DoubleType(), True),
    StructField("pression",       DoubleType(), True),
])

df_meteo_raw = (
    spark.read
    .option("sep", ";")
    .option("header", True)
    .schema(schema_meteo)
    .csv(str(METEO_CSV))
)

print(f"Lignes meteo brutes : {df_meteo_raw.count():,}")
df_meteo_raw.show(5)

Lignes meteo brutes : 2,880
+----------------+-----------+--------------+-------+--------+--------+
|      horodatage|temperature|precipitations|vent_ms|humidite|pression|
+----------------+-----------+--------------+-------+--------+--------+
|2020-11-01T00:00|       14.5|           0.0|   4.22|    87.0|  1008.1|
|2020-11-01T01:00|       14.1|           0.0|   4.09|    84.0|  1008.3|
|2020-11-01T02:00|       13.3|           0.0|   3.96|    87.0|  1008.3|
|2020-11-01T03:00|       13.3|           0.0|   3.41|    87.0|  1008.1|
|2020-11-01T04:00|       13.1|           0.1|   2.83|    90.0|  1008.0|
+----------------+-----------+--------------+-------+--------+--------+
only showing top 5 rows


In [14]:
from pyspark.sql.functions import to_timestamp, col, when, round as spark_round

# Renommage et conversions d'unites
# - temperature : deja en Celsius (Open-Meteo)
# - vent_ms (m/s) converti en vent_kmh (* 3.6)
# - precipitations -> precipitation_mm
df_meteo = (
    df_meteo_raw
    .withColumn("horodatage_meteo", to_timestamp(col("horodatage"), "yyyy-MM-dd'T'HH:mm"))
    .withColumnRenamed("temperature",    "temperature_c")
    .withColumn("vent_kmh",  spark_round(col("vent_ms") * 3.6, 2))
    .withColumnRenamed("humidite",       "humidite_pct")
    .withColumnRenamed("precipitations", "precipitation_mm")
    .select("horodatage_meteo", "temperature_c", "humidite_pct", "vent_kmh", "precipitation_mm")
    .dropna(subset=["horodatage_meteo"])
)

# Indicateur pluie : True si precipitation > 0
df_meteo = df_meteo.withColumn(
    "est_pluie",
    when(col("precipitation_mm") > 0, True).otherwise(False)
)

print(f"Lignes meteo nettoyees : {df_meteo.count():,}")
df_meteo.show(8)
df_meteo.describe(["temperature_c", "humidite_pct", "vent_kmh", "precipitation_mm"]).show()

Lignes meteo nettoyees : 2,880
+-------------------+-------------+------------+--------+----------------+---------+
|   horodatage_meteo|temperature_c|humidite_pct|vent_kmh|precipitation_mm|est_pluie|
+-------------------+-------------+------------+--------+----------------+---------+
|2020-11-01 00:00:00|         14.5|        87.0|   15.19|             0.0|    false|
|2020-11-01 01:00:00|         14.1|        84.0|   14.72|             0.0|    false|
|2020-11-01 02:00:00|         13.3|        87.0|   14.26|             0.0|    false|
|2020-11-01 03:00:00|         13.3|        87.0|   12.28|             0.0|    false|
|2020-11-01 04:00:00|         13.1|        90.0|   10.19|             0.1|     true|
|2020-11-01 05:00:00|         13.1|        94.0|    10.3|             0.6|     true|
|2020-11-01 06:00:00|         14.0|        92.0|   16.13|             0.5|     true|
|2020-11-01 07:00:00|         14.5|        94.0|   17.93|             0.0|    false|
+-------------------+-------------

---
## 2.6 Jointure temporelle : Velib' x Meteo

Les observations Velib' sont a la minute, les donnees SYNOP a l'heure.
Pour joindre les deux tables, on **tronque** les horodatages Velib' a l'heure la plus
proche, puis on effectue une jointure sur cette cle commune.


In [15]:
from pyspark.sql.functions import date_trunc, broadcast, col

# ── 1. Cle de jointure : heure tronquee ──────────────────────────────────────
# On tronque le timestamp Velib' a l'heure pile (ex: 12:37 -> 12:00)
# pour pouvoir le joindre avec la meteo horaire
df_velib_join = df_clean.withColumn(
    "heure_tronquee",
    date_trunc("hour", col("timestamp_parsed"))
)

df_meteo_join = df_meteo.withColumnRenamed(
    "horodatage_meteo", "heure_tronquee"
)

# ── 2. Broadcast join ─────────────────────────────────────────────────────────
# La table meteo est petite (2 880 lignes pour 4 mois d'observations horaires).
# Avec broadcast(), Spark envoie une copie complete a chaque executeur,
# ce qui evite entierement le shuffle de la grosse table Velib'.
df_joint = df_velib_join.join(
    broadcast(df_meteo_join),
    on="heure_tronquee",
    how="left"
)

# Verification
n_avant      = df_clean.count()
n_apres      = df_joint.count()
n_meteo_null = df_joint.filter(col("temperature_c").isNull()).count()

print(f"Lignes avant jointure  : {n_avant:,}")
print(f"Lignes apres jointure  : {n_apres:,}")
print(f"Snapshots sans meteo   : {n_meteo_null:,} ({100*n_meteo_null/n_apres:.1f}%)")

Lignes avant jointure  : 5,267,323
Lignes apres jointure  : 5,267,323
Snapshots sans meteo   : 0 (0.0%)


In [16]:
# Examinons le plan d'execution pour verifier que le broadcast est bien utilise
print("=== Plan d'execution de la jointure ===")
df_joint.explain(mode="formatted")
# Cherchez "BroadcastHashJoin" dans le plan -- pas de "SortMergeJoin" (qui implique un shuffle)

=== Plan d'execution de la jointure ===
== Physical Plan ==
AdaptiveSparkPlan (13)
+- Project (12)
   +- BroadcastHashJoin LeftOuter BuildRight (11)
      :- Project (6)
      :  +- Project (5)
      :     +- Project (4)
      :        +- Project (3)
      :           +- Filter (2)
      :              +- Scan parquet  (1)
      +- BroadcastExchange (10)
         +- Project (9)
            +- Filter (8)
               +- Scan csv  (7)


(1) Scan parquet 
Output [10]: [horodatage#48, capacite#49, velos_disponibles#50, bornettes_libres#51, nom_station#52, coordonnees#53, is_installed#54, timestamp#56, annee#57, mois#58]
Batched: true
Location: InMemoryFileIndex [file:/Users/ms/Library/CloudStorage/SynologyDrive-cefedemaura/cours/hetic/spark_project/data/velib/parquet]
PushedFilters: [IsNotNull(capacite), GreaterThan(capacite,0)]
ReadSchema: struct<horodatage:string,capacite:int,velos_disponibles:int,bornettes_libres:int,nom_station:string,coordonnees:string,is_installed:boolean,timestamp

In [17]:
# [EXERCICE]
# Calculez, pour chaque combinaison (arrondissement, est_pluie),
# le taux_occupation moyen et le nombre de snapshots.
# Triez par arrondissement croissant, puis par est_pluie.
# ──────────────────────────────────────────────────────────────────────────

# Notre df_joint n'a pas de colonne arrondissement.
# On charge stations_info.csv pour recuperer code_arr (arrondissement),
# puis on joint sur nom_station.
df_stations = (
    spark.read
    .option("sep", ";")
    .option("header", True)
    .csv(str(STATIONS_CSV))
    .select("name", "code_arr")
    .withColumnRenamed("name", "nom_station")
)

df_avec_arr = df_joint.join(broadcast(df_stations), on="nom_station", how="left")

(
    df_avec_arr
    .groupBy("code_arr", "est_pluie")
    .agg(
        F.round(F.mean("taux_occupation"), 3).alias("taux_moyen"),
        F.count("*").alias("nb_snapshots")
    )
    .orderBy(F.col("code_arr").cast("int"), "est_pluie")
    .show(40)
)

+--------+---------+----------+------------+
|code_arr|est_pluie|taux_moyen|nb_snapshots|
+--------+---------+----------+------------+
|    NULL|    false|      0.22|      347457|
|    NULL|     true|     0.219|      122337|
|       6|    false|     0.298|       36452|
|       6|     true|     0.311|       12857|
|       7|    false|     0.264|       19628|
|       7|     true|     0.267|        6923|
|      29|    false|     0.375|        2804|
|      29|     true|     0.376|         989|
|      30|    false|      0.25|        5608|
|      30|     true|     0.262|        1978|
|      36|    false|     0.274|       14020|
|      36|     true|     0.279|        4945|
|      37|    false|      0.32|       11216|
|      37|     true|     0.314|        3956|
|      38|    false|      0.16|       16824|
|      38|     true|     0.157|        5934|
|      52|    false|     0.071|       14020|
|      52|     true|     0.081|        4945|
|      63|    false|     0.159|        5608|
|      63|

---
## 2.7 Persistance en memoire : `.cache()` et `.persist()`

Nous allons utiliser `df_joint` intensivement pendant le reste du cours.
Mettons-le en cache pour eviter de recalculer la jointure a chaque action.


In [18]:
# .cache() == .persist(StorageLevel.MEMORY_AND_DISK)
# Si les donnees ne tiennent pas en RAM, Spark deborde sur disque.

t0 = time.perf_counter()
df_joint.cache()
df_joint.count()   # force la materialisation du cache
t_cache = time.perf_counter() - t0
print(f"Mise en cache (premiere passe) : {t_cache:.2f} s")

# Deuxieme passe -- depuis le cache
t0 = time.perf_counter()
df_joint.count()
t_hit = time.perf_counter() - t0
print(f"Lecture depuis le cache        : {t_hit:.2f} s")
print(f"Gain                           : x{t_cache/t_hit:.1f}")
print()
print("Allez dans Spark UI -> Storage pour voir la taille du cache.")

Mise en cache (premiere passe) : 3.99 s
Lecture depuis le cache        : 0.10 s
Gain                           : x40.4

Allez dans Spark UI -> Storage pour voir la taille du cache.


In [19]:
# StorageLevel disponibles (par ordre de rapidite / consommation memoire)
from pyspark import StorageLevel

print("StorageLevels disponibles :")
print("  MEMORY_ONLY          : RAM uniquement (eviction si plein)")
print("  MEMORY_AND_DISK      : RAM, puis disque si plein  [defaut de .cache()]")
print("  DISK_ONLY            : Disque uniquement (lent mais stable)")
print("  MEMORY_ONLY_SER      : RAM, serialise (moins de RAM, plus de CPU)")
print("  OFF_HEAP             : Memoire hors JVM (necessite configuration)")
print()
print("Regle pratique :")
print("  - Petits DataFrames reutilises souvent -> MEMORY_ONLY")
print("  - Gros DataFrames reutilises -> MEMORY_AND_DISK")
print("  - Ne pas cacher si utilise une seule fois -> overhead inutile")

StorageLevels disponibles :
  MEMORY_ONLY          : RAM uniquement (eviction si plein)
  MEMORY_AND_DISK      : RAM, puis disque si plein  [defaut de .cache()]
  DISK_ONLY            : Disque uniquement (lent mais stable)
  MEMORY_ONLY_SER      : RAM, serialise (moins de RAM, plus de CPU)
  OFF_HEAP             : Memoire hors JVM (necessite configuration)

Regle pratique :
  - Petits DataFrames reutilises souvent -> MEMORY_ONLY
  - Gros DataFrames reutilises -> MEMORY_AND_DISK
  - Ne pas cacher si utilise une seule fois -> overhead inutile


---
## 2.8 Ecriture en Parquet partitionne

La table `df_joint` est la table centrale du projet. Nous allons la persister
sur disque en Parquet, partitionne par annee et par mois, pour que les
traitements des jours suivants n'aient pas a la reconstruire.


In [20]:
OUTPUT_VELIB_CONSOLIDE = OUTPUT_DIR / "disponibilite_consolidee.parquet"

# Selection finale des colonnes (on elimine les colonnes intermediaires)
colonnes_finales = [
    "nom_station", "capacite",
    "horodatage", "velos_disponibles", "bornettes_libres",
    "taux_occupation", "statut",
    "annee", "mois", "jour_sem", "heure", "is_weekend",
    "temperature_c", "humidite_pct", "vent_kmh", "precipitation_mm", "est_pluie"
]

df_sortie = df_joint.select(*colonnes_finales)

t0 = time.perf_counter()
(
    df_sortie.write
    .mode("overwrite")
    .partitionBy("annee", "mois")
    .parquet(str(OUTPUT_VELIB_CONSOLIDE))
)
t_write = time.perf_counter() - t0
print(f"Ecriture en {t_write:.1f} s")

# Verification : lecture selective d'une partition
df_verif = spark.read.parquet(str(OUTPUT_VELIB_CONSOLIDE)).filter("annee = 2020 AND mois = 11")
print(f"Novembre 2020 : {df_verif.count():,} lignes")
print(f"Colonnes : {df_verif.columns}")

26/07/23 12:17:26 WARN MemoryManager: Total allocation exceeds 95,00% (1 020 054 720 bytes) of heap memory
Scaling row group sizes to 95,00% for 8 writers
26/07/23 12:17:26 WARN MemoryManager: Total allocation exceeds 95,00% (1 020 054 720 bytes) of heap memory
Scaling row group sizes to 95,00% for 8 writers
26/07/23 12:17:26 WARN MemoryManager: Total allocation exceeds 95,00% (1 020 054 720 bytes) of heap memory
Scaling row group sizes to 84,44% for 9 writers
26/07/23 12:17:26 WARN MemoryManager: Total allocation exceeds 95,00% (1 020 054 720 bytes) of heap memory
Scaling row group sizes to 95,00% for 8 writers
26/07/23 12:17:26 WARN MemoryManager: Total allocation exceeds 95,00% (1 020 054 720 bytes) of heap memory
Scaling row group sizes to 84,44% for 9 writers
26/07/23 12:17:26 WARN MemoryManager: Total allocation exceeds 95,00% (1 020 054 720 bytes) of heap memory
Scaling row group sizes to 95,00% for 8 writers


Ecriture en 2.6 s
Novembre 2020 : 539,932 lignes
Colonnes : ['nom_station', 'capacite', 'horodatage', 'velos_disponibles', 'bornettes_libres', 'taux_occupation', 'statut', 'jour_sem', 'heure', 'is_weekend', 'temperature_c', 'humidite_pct', 'vent_kmh', 'precipitation_mm', 'est_pluie', 'annee', 'mois']


In [21]:
# Comparaison de la taille CSV vs Parquet
VELIB_RAW_DIR = DATA_DIR / "velib" / "raw"

def taille_dossier_mb(path: Path) -> float:
    total = sum(fichier.stat().st_size for fichier in path.rglob("*") if fichier.is_file())
    return total / (1024 * 1024)

t_csv  = taille_dossier_mb(VELIB_RAW_DIR)
t_parq = taille_dossier_mb(OUTPUT_VELIB_CONSOLIDE)
ratio  = t_csv / t_parq if t_parq > 0 else 0

print(f"Taille CSV bruts            : {t_csv:.1f} MB")
print(f"Taille Parquet consolide    : {t_parq:.1f} MB")
print(f"Rapport CSV/Parquet         : x{ratio:.1f}")
print()
print("Le Parquet est plus petit car :")
print("  1. Stockage colonnaire : on compresse des valeurs homogenes")
print("  2. Encodage par dictionnaire pour les strings repetitives (noms de stations)")
print("  3. Run-Length Encoding sur les colonnes de partition (annee, mois)")

Taille CSV bruts            : 376.2 MB
Taille Parquet consolide    : 26.5 MB
Rapport CSV/Parquet         : x14.2

Le Parquet est plus petit car :
  1. Stockage colonnaire : on compresse des valeurs homogenes
  2. Encodage par dictionnaire pour les strings repetitives (noms de stations)
  3. Run-Length Encoding sur les colonnes de partition (annee, mois)


---
## Bilan du Jour 1

### Ce que nous avons fait

| Etape | API | Concept cle |
|-------|-----|-------------|
| Chargement CSV brut | RDD | `sc.textFile()`, parsing manuel |
| Filtrage et transformation | RDD | `map()`, `filter()`, evaluation paresseuse |
| Agregation par cle | RDD | `reduceByKey()` vs `groupByKey()` |
| Profil horaire | RDD | `flatMap()`, calcul de moyenne distribuee |
| Jointure RDD | RDD | `join()` sur paires (cle, valeur) |
| Lecture Parquet | DataFrame | `spark.read.parquet()`, predicate pushdown |
| Exploration statistique | DataFrame | `describe()`, comptage de nulls |
| Nettoyage | DataFrame | `dropna()`, `when()`, `greatest()` |
| Features temporelles | DataFrame | `year()`, `hour()`, `dayofweek()` |
| Jointure broadcast | DataFrame | `broadcast()`, eviter le shuffle |
| Persistance | DataFrame/RDD | `.cache()`, `.persist()`, `StorageLevel` |
| Ecriture partitionnee | DataFrame | `write.partitionBy().parquet()` |

### Concepts fondamentaux a retenir

1. **Evaluation paresseuse** : les transformations construisent un plan, les actions l'executent.
2. **Le DAG** : lire le Spark UI est une competence essentielle, pas optionnelle.
3. **reduceByKey > groupByKey** : toujours combiner localement avant de shuffler.
4. **DataFrame > RDD** : l'optimiseur Catalyst rend les DataFrames significativement plus
   rapides que les RDD pour les memes operations.
5. **Cache strategique** : ne mettre en cache que ce qui est reutilise plusieurs fois.
6. **Broadcast join** : quand une table est petite, eviter le shuffle de la grosse table.

### Pour demain (Jour 2)

La table `disponibilite_consolidee.parquet` sera le point de depart du Jour 2.
Nous allons l'interroger avec Spark SQL (fenetrage temporel, requetes analytiques),
puis la connecter a un flux simule de mises a jour en temps reel avec Structured Streaming.


In [22]:
# Nettoyage de fin de session
# Toujours liberer la SparkSession proprement pour liberer les ressources
print("Arret de la SparkSession...")
spark.stop()
print("SparkSession arretee. Bonne nuit !")

Arret de la SparkSession...
SparkSession arretee. Bonne nuit !
